# Ćwiczenia 5 — Walidacja krzyżowa i przeciek (wersja Python)
**Dane demo: kohorta T2D (QinJ_2012)**

Na Ćwiczeniach 4 model k najbliższych sąsiadów (k-NN) oceniono na jednym zbiorze testowym, a dokładność zmieniała się wyraźnie po zmianie ziarna podziału. Przy kilkudziesięciu próbkach testowych jedna liczba zależy więc w dużej mierze od tego, które próbki trafiły do testu. Na tym spotkaniu poznajemy sposób oceny, który wykorzystuje wszystkie próbki treningowe i mniej zależy od przypadku: walidację krzyżową. Na koniec omawiamy przeciek danych, czyli najczęstszy błąd, który zawyża ocenę modelu.

Przez całe spotkanie model jest ten sam co na Ćwiczeniach 4: k-NN na standaryzowanych cechach CLR. Zmienia się tylko sposób jego oceny. Zbiór testowy odkładamy i nie używamy go: wszystkie oceny pochodzą z walidacji krzyżowej na zbiorze treningowym. Liczby różnią się nieco od wersji R, bo oba języki inaczej losują podział na części; wnioski są takie same.

## Punkt wyjścia

Komórka odtwarza przygotowanie danych z Ćwiczeń 4: podział warstwowy z ziarnem 42, filtr częstości występowania wyznaczony na treningu (zostają cechy obecne w co najmniej 10% próbek treningowych) i przekształcenie CLR. Notatnik najlepiej uruchamiać od początku po ponownym uruchomieniu jądra (*Kernel → Restart*), komórka po komórce: obiekty pozostawione w pamięci przez wcześniejszy kod mogą mieć te same nazwy i inną zawartość.

In [ ]:
import numpy as np                       # obliczenia na tablicach
import pandas as pd                      # tabele danych
import matplotlib.pyplot as plt          # wykresy
from sklearn.model_selection import train_test_split   # podział na trening i test
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold   # części walidacji krzyżowej
from sklearn.model_selection import cross_val_score, GridSearchCV   # walidacja krzyżowa i strojenie
from sklearn.pipeline import make_pipeline             # potok: kroki i model w jednym obiekcie
from sklearn.preprocessing import StandardScaler       # standaryzacja
from sklearn.neighbors import KNeighborsClassifier     # k najbliższych sąsiadów
pd.set_option("display.width", 120)      # szersze wydruki tabel

X = pd.read_csv("dane/t2d_features.csv", index_col=0)   # index_col=0: pierwsza kolumna to nazwy próbek
# ["grupa"]: kolumna z etykietami; index=X.index: te same nazwy wierszy co w X
y = pd.Series(pd.read_csv("dane/t2d_labels.csv")["grupa"].to_numpy(), index=X.index)
# test_size=0.25: 25% próbek do testu; stratify=y: osobno w każdej klasie; random_state=42: ziarno kursu
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

prevalence_tr = (X_tr > 0).mean()        # częstość występowania każdej cechy w treningu
keep = prevalence_tr.index[prevalence_tr >= 0.10]   # nazwy cech obecnych w co najmniej 10% próbek
X_tr_f = X_tr[keep]                      # trening: tylko zachowane cechy

# CLR: logarytm (z dodaną stałą 1e-6, bo log(0) nie istnieje) minus średnia logarytmów w próbce
def clr(M):
    L = np.log(M + 1e-6)                 # logarytm każdej wartości
    return L.sub(L.mean(axis=1), axis=0) # od każdego wiersza (próbki) odejmujemy jego średnią

Z_tr = clr(X_tr_f)                       # cechy treningowe po przekształceniu CLR

print(Z_tr.shape)                        # liczba próbek treningowych i liczba cech
print(len(y_te))                         # liczba próbek odłożonych do testu (X_te, y_te: nie używamy do Ćwiczeń 9)
print(y_tr.value_counts())               # liczba próbek każdej klasy w treningu
baseline = y_tr.value_counts(normalize=True).max()   # klasyfikator bazowy: udział klasy najliczniejszej
print(baseline)                          # dokładność klasyfikatora bazowego

**Na co patrzeć.** Klasyfikator bazowy, który każdej próbce przypisuje klasę najliczniejszą, osiąga dokładność około 0,5. Każdy wynik walidacji krzyżowej porównujemy z tą liczbą.

## 1. Walidacja krzyżowa k-krotna

Walidacja krzyżowa (ang. *cross-validation*, CV) dzieli zbiór treningowy na k równych części (ang. *folds*). Każda część raz pełni rolę części sprawdzającej, a model uczy się wtedy na pozostałych k − 1 częściach (część ucząca). Oceną modelu jest średnia z k wyników. W porównaniu z jednym zbiorem testowym każda próbka treningowa raz służy do oceny, więc wynik opiera się na wszystkich próbkach, a nie na kilkudziesięciu. Części losuje się osobno w każdej klasie (podział warstwowy, ang. *stratified*), żeby proporcje klas były w nich takie jak w całym zbiorze.

Często spotyka się też podział na trzy zbiory: treningowy, walidacyjny (do wyboru hiperparametrów) i testowy. Walidacja krzyżowa zastępuje jeden zbiór walidacyjny: każda część sprawdzająca jest po kolei takim zbiorem walidacyjnym. Przy kilkuset próbkach jeden zbiór walidacyjny liczyłby około 50 próbek, a wynik zależałby od tego, które trafiły do niego (jak na Ćwiczeniach 4); stały zbiór walidacyjny stosuje się przy dużych zbiorach danych.

Najpierw wykonujemy ręcznie jedno powtórzenie, żeby było widać, co się w nim dzieje: część 1 jest sprawdzająca, a model uczy się na pozostałych czterech. Model to ten sam potok co na Ćwiczeniach 4: standaryzacja i k-NN.

In [ ]:
# 5 części warstwowych; shuffle=True: próbki tasowane przed podziałem; random_state=42: ten sam podział za każdym razem
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
parts = list(skf.split(Z_tr, y_tr))      # lista 5 par: (numery próbek części uczącej, numery części sprawdzającej)
print([len(fit_idx) for fit_idx, val in parts])   # liczba próbek uczących w każdym powtórzeniu

fit_idx, val = parts[0]                  # powtórzenie 1: część ucząca i część sprawdzająca
print(y_tr.iloc[val].value_counts())     # część sprawdzająca: obie klasy prawie po równo

# potok: standaryzacja, potem k-NN z 5 sąsiadami; .fit() dopasowuje oba kroki tylko na części uczącej
knn_1 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))   # potok jak na Ćwiczeniach 4
knn_1.fit(Z_tr.iloc[fit_idx], y_tr.iloc[fit_idx])   # uczenie tylko na części uczącej
print(knn_1.score(Z_tr.iloc[val], y_tr.iloc[val]))   # dokładność w części 1

**Na co patrzeć.** Proporcji nie podaje się, bo wynika z liczby części: przy 5 częściach każda to jedna piąta treningu, więc w każdym powtórzeniu model uczy się na 80% próbek treningowych (około 207) i jest sprawdzany na pozostałych 20% (około 52). Zbiór testowy z Ćwiczeń 4 nie bierze w tym udziału: wszystkie części powstają z samego treningu. Standaryzację wyznaczamy tylko na części uczącej: część sprawdzająca ma udawać nowe próbki, więc nie może wpływać na żaden krok uczenia.

To samo dla wszystkich pięciu części robi jedno wywołanie `cross_val_score()`. Argument `cv` przekazuje części z komórki powyżej:

In [ ]:
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))   # potok: standaryzacja + k-NN
scores = cross_val_score(model, Z_tr, y_tr, cv=skf)   # cv=skf: te same 5 części co wyżej
print(scores.round(3))                   # dokładność w każdej części
print(scores.mean(), scores.std(ddof=1)) # średnia z 5 części i odchylenie standardowe (ddof=1: jak w R)

**Na co patrzeć.** `cross_val_score()` dopasował potok pięć razy, za każdym razem na innych czterech częściach, i sprawdził go na piątej. Pierwszy wynik jest taki sam jak w ręcznym powtórzeniu. `cross_val_score()` służy tylko do oceny: model do przewidywań dopasowuje się potem raz, na całym treningu (`.fit(Z_tr, y_tr)`). Pięć dokładności różni się między sobą, choć model i dane są te same, bo każda część liczy tylko około 52 próbek. Średnia z pięciu części jest oceną modelu, a odchylenie standardowe mierzy rozrzut wyników.

🔧 **Do sprawdzenia.** Zmienić `random_state` w `StratifiedKFold()` (np. na `1`) i uruchomić obie komórki ponownie. Jak zmieniają się dokładności w częściach i ich średnia?

> **✏️ Zadanie 1.** Policzyć różnicę między największą a najmniejszą dokładnością w częściach (`.max()` i `.min()` na tablicy `scores`). Dlaczego jakość modelu opisuje się średnią z części, a nie wynikiem najlepszej części?

In [ ]:
# miejsce na kod do Zadania 1

## 2. Liczba części

Liczba części wpływa na to, z ilu próbek liczy się pojedynczy wynik. Walidacja k-krotna to
walidacja na k częściach: model dopasowuje się k razy, za każdym razem uczy się na k − 1
częściach i jest sprawdzany na pozostałej. Przy około 258 próbkach treningowych:

| Walidacja | Część ucząca | Część sprawdzająca | Liczba wyników |
|-----------|--------------|--------------------|----------------|
| 3-krotna  | 2/3 (około 173 próbki) | 1/3 (około 86) | 3 |
| 5-krotna  | 4/5 (około 207) | 1/5 (około 52) | 5 |
| 10-krotna | 9/10 (około 233) | 1/10 (około 26) | 10 |

Porównujemy te trzy warianty dla tego samego modelu.

In [ ]:
scores_3 = cross_val_score(model, Z_tr, y_tr, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42))    # walidacja 3-krotna
scores_10 = cross_val_score(model, Z_tr, y_tr, cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=42))  # walidacja 10-krotna

acc_parts = {"3 części": scores_3, "5 części": scores, "10 części": scores_10}   # dokładności z części (5 części: sekcja 1)
# tabela: dla każdego wariantu liczba wyników, ich średnia i odchylenie standardowe
print(pd.DataFrame({"results": [len(s) for s in acc_parts.values()],          # liczba wyników (części)
                    "mean": [s.mean() for s in acc_parts.values()],             # średnia dokładność
                    "sd": [s.std(ddof=1) for s in acc_parts.values()]},         # odchylenie standardowe
                   index=acc_parts.keys()).round(3))                            # wiersze: warianty walidacji


**Na co patrzeć.** Średnie są podobne, bo model jest wszędzie ten sam. Przy 10 częściach każda część sprawdzająca liczy około 26 próbek, więc pojedynczy wynik jest mniej dokładny niż przy 3 częściach (około 86 próbek), a wyniki z części bardziej się rozrzucają.

Wynik zależy też od tego, jak wylosowano części. Walidacja powtarzana (ang. *repeated cross-validation*) losuje podział na części kilka razy i uśrednia wszystkie wyniki, więc średnia mniej zależy od jednego losowania. W scikit-learn służy do tego `RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)`. Skrajny wariant to LOOCV (ang. *leave-one-out*, `LeaveOneOut()`): każda część to jedna próbka, więc model dopasowuje się tyle razy, ile jest próbek; stosuje się go tylko dla szybkich modeli.

> **✏️ Zadanie 2.** Ocenić ten sam model walidacją 5-krotną powtórzoną 3 razy (`cross_val_score()` jak dla `scores_3`, z `RepeatedStratifiedKFold` podanym wyżej). Ile wyników uśredniono (`len()`) i jaka jest średnia dokładność? Czy różni się wyraźnie od walidacji 5-krotnej z sekcji 1?

In [ ]:
# miejsce na kod do Zadania 2

## 3. Dobór liczby sąsiadów

Liczba sąsiadów k w k-NN to hiperparametr (ang. *hyperparameter*): ustala się go przed uczeniem, a model sam go nie wyznacza. Walidacja krzyżowa pozwala porównać kilka wartości k na tych samych częściach i wybrać tę, która daje najlepszą średnią dokładność. Takie porównywanie nazywa się strojeniem (ang. *tuning*). W scikit-learn robi to `GridSearchCV()`: podaje się potok i listę wartości hiperparametru.

In [ ]:
# siatka: nazwa kroku w potoku (kneighborsclassifier), dwa podkreślniki, nazwa hiperparametru; k = 1, 3, ..., 31
grid = {"kneighborsclassifier__n_neighbors": list(range(1, 32, 2))}
tune_model = GridSearchCV(model, grid, cv=skf).fit(Z_tr, y_tr)   # te same 5 części co w sekcji 1

res = pd.DataFrame({"k": grid["kneighborsclassifier__n_neighbors"],   # tabela wyników strojenia
                    "Accuracy": tune_model.cv_results_["mean_test_score"],    # średnia dokładność z 5 części
                    "AccuracySD": tune_model.cv_results_["std_test_score"]})  # odchylenie standardowe
print(res.round(3))                      # wynik dla każdej wartości k
print(tune_model.best_params_)           # k o największej średniej dokładności

ax = res.plot(x="k", y="Accuracy", marker="o", legend=False)   # średnia dokładność dla każdego k
ax.axhline(baseline, linestyle="--", color="grey")   # linia przerywana: klasyfikator bazowy
plt.show()                               # wyświetlenie wykresu

**Na co patrzeć.** Wiersz dla k = 5 ma ten sam wynik co w sekcji 1, bo części są te same. (Odchylenie standardowe scikit-learn liczy z dzieleniem przez liczbę części, a nie przez liczbę części minus 1, dlatego jest nieco mniejsze niż w sekcji 1). Krzywa jest nieregularna, a różnice między sąsiednimi wartościami k są rzędu kilku punktów procentowych. Najlepsze k wybrano spośród 16 kandydatów, więc jego wynik jest nieco zbyt optymistyczny: część przewagi wynika z przypadku. Do tego problemu wracamy na Ćwiczeniach 9.

> **✏️ Zadanie 3.** Wypisać z tabeli `res` wiersze dla k = 5 i dla najlepszego k (np. `res[res["k"] == 5]`) i policzyć różnicę średnich dokładności. Czy jest większa od odchylenia standardowego wyników z części (`AccuracySD`)? Czy najlepsze k jest wyraźnie lepsze od k = 5?

In [ ]:
# miejsce na kod do Zadania 3

## 4. Przeciek danych

Przeciek danych (ang. *data leakage*) zachodzi, gdy informacja z próbek, na których oceniamy model, wpływa na jego uczenie. Ocena jest wtedy zawyżona, a model na nowych danych działa gorzej, niż obiecuje walidacja. Zasada jest jedna: każdy krok, który wyznacza coś na podstawie wielu próbek albo etykiet, należy do modelu i musi być wykonywany w każdej części walidacji od nowa, tylko na części uczącej. Dla każdego kroku przygotowania danych należy zapytać, czy jego wynik zależy od innych próbek niż ta, którą przekształca.

| Krok | Czy wyznacza coś z danych? | Miejsce |
|------|---------------------------|---------|
| podział na zbiór treningowy i testowy | — | na samym początku |
| CLR, logarytm ze stałą | nie (każda próbka osobno) | może poprzedzać walidację |
| filtr częstości występowania | tak, bez etykiet | wewnątrz każdej części |
| standaryzacja, imputacja braków | tak, bez etykiet | wewnątrz każdej części (krok w potoku) |
| wybór cech według związku z klasą | tak, z etykietami | wewnątrz każdej części (krok w potoku) |
| dobór hiperparametrów (sekcja 3) | tak, z etykietami | wewnątrz walidacji; ocena wybranego modelu na Ćwiczeniach 9 |

Najgroźniejszy jest wybór cech według etykiet wykonany przed walidacją, np. wybranie 20 taksonów najbardziej różniących klasy na całym zbiorze treningowym, a dopiero potem walidacja krzyżowa na tych taksonach. Części sprawdzające brały wtedy udział w wyborze cech, więc model „widział” je wcześniej. Przy setkach cech i kilkuset próbkach zawsze znajdzie się kilka cech, które przypadkiem dobrze różnicują klasy, i taka procedura daje wynik wyraźnie powyżej klasyfikatora bazowego nawet wtedy, gdy etykiety nie mają żadnego związku z cechami. W scikit-learn wybór cech umieszcza się w potoku (np. `make_pipeline(SelectKBest(k=20), StandardScaler(), KNeighborsClassifier())`), a wtedy `cross_val_score()` powtarza go w każdej części.

Kroki, które nie korzystają z etykiet, dają przeciek znacznie mniejszy. W tym notatniku filtr częstości występowania wyznaczono raz, na całym zbiorze treningowym; w pełni poprawna procedura umieszcza go wewnątrz każdej części. Jeśli kilka próbek pochodzi od tej samej osoby, wszystkie muszą trafić do tej samej części; inaczej model jest oceniany na osobie, którą już widział (w scikit-learn służy do tego `GroupKFold`).

> **✏️ Zadanie 4.** Sprawdzić, jak duży jest przeciek przy standaryzacji. Wystandaryzować cały zbiór treningowy przed walidacją (`Z_std = StandardScaler().fit_transform(Z_tr)`), a potem ocenić sam k-NN (`KNeighborsClassifier(n_neighbors=5)`, bez potoku) na `Z_std`, na tych samych częściach (`cv=skf`). Porównać średnią dokładność z wynikiem z sekcji 1. Czy różnica jest duża w porównaniu z odchyleniem standardowym wyników z części? Dlaczego przeciek jest tu niewielki?

In [ ]:
# miejsce na kod do Zadania 4